In [1]:
from dotenv import load_dotenv
import os 
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv('../../env', override=True)
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
print(AZURE_OPENAI_API_KEY[:10])
print(MODEL_NAME)

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ['LANGCHAIN_TRACING_V2'] = 'false' #true, false
os.environ['LANGCHAIN_PROJECT'] = 'AGENT'

if os.getenv('LANGCHAIN_TRACING_V2') == "true":
    if len(os.getenv('LANGCHAIN_API_KEY')) > 0:
        print('랭스미스로 추적 중입니다 :', os.getenv('LANGSMITH_API_KEY')[:10])
    else:
        print('랭스미스 키가 확인되지 않았습니다.')

Em4f4T5UbT
gpt-5-nano


In [2]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    model=os.environ['MODEL_NAME'],
    azure_deployment=os.environ["MODEL_NAME"],
    azure_endpoint=os.environ["END_POINT"],
    openai_api_version=os.environ["MODEL_API_VERSION"],
    openai_api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

In [3]:
# 아래 코드에서 stdio로 실행하기 때문에 현재는 필요없고 streamable_http로 실행하기 위해서는 주석을 해제해야 합니다.
# import subprocess

# port = 8001
# try:
#     subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)
# except Exception:
#     pass

# MCP_URL = os.getenv("MCP_URL", f"http://localhost:{port}/mcp")

# import sys, subprocess
# print("using python:", sys.executable)
# subprocess.Popen([sys.executable, "mcp_calc_server.py"])

# print("MCP 서버 URL:", MCP_URL)

In [4]:
import asyncio
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.checkpoint.memory import MemorySaver
import sys

SERVERS = {
    "math": {
        "transport": "stdio",
        "command": sys.executable,
        "args": ["./mcp_calc_server.py"],
    },
}

client = MultiServerMCPClient(SERVERS)

client_tools = await client.get_tools()
client_tools

[StructuredTool(name='add', description='덧셈 : 두 수를 더합니다. a + b', args_schema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x116d8eac0>),
 StructuredTool(name='subtract', description='뺄셈 : 두 수를 뺍니다. a - b', args_schema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x1176b0e00>),
 StructuredTool(name='multiply', description='곱셈 : 두 수를 곱합니다. a * b', args_schema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a'

In [5]:
from langchain_core.tools import tool

@tool
async def info(_: None = None) -> str:
    """앱의 정보를 반환합니다"""
    out = await client.get_resources("math", uris = "data://config")
    return out[0].data
    
@tool
async def user_profile(user_id: int) -> str:
    """사용자의 프로필을 반환합니다."""
    out = await client.get_resources("math", uris = f"users://{user_id}/profile")
    return out[0].data

tools = client_tools + [info, user_profile]
tools

[StructuredTool(name='add', description='덧셈 : 두 수를 더합니다. a + b', args_schema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x116d8eac0>),
 StructuredTool(name='subtract', description='뺄셈 : 두 수를 뺍니다. a - b', args_schema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x1176b0e00>),
 StructuredTool(name='multiply', description='곱셈 : 두 수를 곱합니다. a * b', args_schema={'additionalProperties': False, 'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a'

In [6]:
await info.ainvoke({}) #내부에서는 반드시 input 인자를 필요로 하므로 빈 dict를 넣어줌줌

'{"name": "basic_calculator", "description": "앱의 구성요소를 제공합니다.", "version": "0.0.1", "author": "Kim jintae", "email": "jt.kim@fake.com"}'

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

SYSTEM = """너는 한국어 계산 챗봇 에이전트다.
- 네가 가진 도구를 써서 대답할 수 있으면 대답하고, 아니면 거절하라.
- 앱의 정보는 info 툴을 호출해 얻을 수 있다.
- 사용자의 정보는 user_info 툴을 호출해 얻을 수 있다.
- 사용자의 프로필은 사용자의 아이디를 알면 user_profile 툴을 호출해 얻을 수 있다."
- 만약 사용자의 아이디를 알려주지 않거나 해당 아이디의 유저가 없다면 default 유저 아이디는 100이다. 
- 사용자의 자연어 요청을 이해하고, 필요한 경우 계산 툴을 사용할 수 있다.
- 계산 툴은 다음과 같다. : add/subtract/multiply/divide/square/square_root
- 숫자/단위를 명확히 제시하고, 계산 과정을 간단히 설명하라(필요시).

- 음수의 제곱근(root)을 요구할 때에는 양수의 제곱근(root)을 계산하고 난 후 뒤에 i를 붙여 반환하라.
"""

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM,
    debug=True,                   
)

In [ ]:
async def run(query: str):
    thread_cfg = {"configurable": {"thread_id": "101"}}
    user_msg = {"messages": [{"role": "user", "content": query}]}
    result = await agent.ainvoke(user_msg, config=thread_cfg)
    last = result["messages"][-1]
    print(last.content)

In [ ]:
await run("3+5는?")

[values] {'messages': [HumanMessage(content='3+5는?', additional_kwargs={}, response_metadata={}, id='0b1713f9-c728-42c5-aad9-9a12f2ce5863')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 155, 'prompt_tokens': 619, 'total_tokens': 774, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DITPwU4ZS558DTXZanXwkLLfjUxx3', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None, 'content_filter_results': {}}, id='lc_run--019ce0a3-c879-7f30-bde0-dc3777ee5de7-0', tool_calls=[{'name': 'add', 'args': {'a': 3, 'b': 5}, 'id': 'call_Mfy34YcNO9xJv4kEMmsQBFZY', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metad

In [10]:
await run("(4 + 5) / (7 -7)?")

[values] {'messages': [HumanMessage(content='(4 + 5) / (7 -7)?', additional_kwargs={}, response_metadata={}, id='a85a0b94-2a62-43a4-88c5-4aa834b9a488')]}
[updates] {'model': {'messages': [AIMessage(content='질문하신 식: (4 + 5) / (7 - 7)\n\n계산 과정:\n- 4 + 5 = 9\n- 7 - 7 = 0\n- 따라서 9 / 0은 정의되지 않습니다. 0으로 나눌 수 없어요.\n\n해결 방법 제안:\n- 분모를 0이 아닌 값으로 바꿔 계산해 드릴 수 있습니다. 예: (4+5)/(7-6) = 9/1 = 9\n- 또는 다른 식으로 계산해 드리길 원하시면 알려주세요.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1098, 'prompt_tokens': 626, 'total_tokens': 1724, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 960, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DITQ2YLG8uMW28TLjUbAX6wzw062N', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None, 'content_filter_resu

In [11]:
await run("-100의 루트는?")

[values] {'messages': [HumanMessage(content='-100의 루트는?', additional_kwargs={}, response_metadata={}, id='49957c94-5aca-41f8-a6c7-f4155c7cf3cf')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 409, 'prompt_tokens': 621, 'total_tokens': 1030, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DITQ9fMHH55pPqJVJWSpj6xEjhbXq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None, 'content_filter_results': {}}, id='lc_run--019ce0a3-fe14-7500-9476-be58c8d4cfa3-0', tool_calls=[{'name': 'square_root', 'args': {'a': 100}, 'id': 'call_TTP8J0BxIyfRzlQeRmHnNMrC', 'type': 'tool_call'}], invalid_tool_calls=[], usa

In [12]:
await run("이 앱을 만든 사람은 누구야?")

[values] {'messages': [HumanMessage(content='이 앱을 만든 사람은 누구야?', additional_kwargs={}, response_metadata={}, id='faa1dc4d-c809-4a5a-b2d7-94d411cf3541')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 147, 'prompt_tokens': 623, 'total_tokens': 770, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DITQI3e8eSfPdH8nzyMnvmlFxlNBd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None, 'content_filter_results': {}}, id='lc_run--019ce0a4-2153-71c3-8177-91581f90f184-0', tool_calls=[{'name': 'info', 'args': {}, 'id': 'call_NVwAL3Ft9Raf23AFErVpdsVr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadat

In [13]:
await run("내 이름이 뭐야?")

[values] {'messages': [HumanMessage(content='내 이름이 뭐야?', additional_kwargs={}, response_metadata={}, id='9f0521d8-fd90-4357-88cb-e9e9f0b956a1')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 858, 'prompt_tokens': 620, 'total_tokens': 1478, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 832, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DITQS0GKLAd3YDD0VsKE3W5WRCTVe', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None, 'content_filter_results': {}}, id='lc_run--019ce0a4-44de-7772-ac01-b525e4571ddc-0', tool_calls=[{'name': 'user_profile', 'args': {'user_id': 100}, 'id': 'call_giC3utKgxfBY89euCTK3rx5X', 'type': 'tool_call'}], invalid_tool_calls=[

In [14]:
await run("내 thread_id가 뭐야?")

[values] {'messages': [HumanMessage(content='내 thread_id가 뭐야?', additional_kwargs={}, response_metadata={}, id='87110493-b70d-4492-ba0d-ca3889a9e905')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 403, 'prompt_tokens': 621, 'total_tokens': 1024, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DITQcHmmJp5Ni3YEdtxwRlyaDV6Ay', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None, 'content_filter_results': {}}, id='lc_run--019ce0a4-6db5-7122-9f0c-ed3241e0e461-0', tool_calls=[{'name': 'info', 'args': {}, 'id': 'call_3ZslYmtrVM1p2ug0qqY2l1fL', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metada

In [15]:
await run("102번 유저의 정보를 알려줘!")

[values] {'messages': [HumanMessage(content='102번 유저의 정보를 알려줘!', additional_kwargs={}, response_metadata={}, id='b3431390-073f-49f6-887e-3f7d9fdba908')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 218, 'prompt_tokens': 623, 'total_tokens': 841, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 192, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DITR7GIP3Vbim7xjkCYszjDsnkaWd', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None, 'content_filter_results': {}}, id='lc_run--019ce0a4-e51e-7033-a16c-14e04610bd96-0', tool_calls=[{'name': 'user_profile', 'args': {'user_id': 102}, 'id': 'call_qPcCQE9bwU747cNfsN7MToOa', 'type': 'tool_call'}], invalid_tool_

In [16]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest, ModelResponse
from langchain.agents.middleware import PIIMiddleware
import re

class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_id_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user id."""
    user_id = request.runtime.context.get("user_id", "100")
    base_prompt = """너는 한국어 계산 챗봇 에이전트다.
- 네가 가진 도구를 써서 대답할 수 있으면 대답하고, 아니면 거절하라.
- 앱의 정보는 info 툴을 호출해 얻을 수 있다.
- 사용자의 프로필은 사용자의 아이디를 알면 user_profile 툴을 호출해 얻을 수 있다."
- 만약 사용자의 아이디를 알려주지 않거나 해당 아이디의 유저가 없다면 default 유저 아이디는 100이다. 
- 사용자의 자연어 요청을 이해하고, 필요한 경우 계산 툴을 사용할 수 있다.
- 계산 툴은 다음과 같다. : add/subtract/multiply/divide/square/square_root
- 숫자/단위를 명확히 제시하고, 계산 과정을 간단히 설명하라(필요시).
- 나눗셈의 0 나누기 등 오류 상황은 정중히 안내하라.
- 음수의 제곱근(root)을 요구할 때에는 양수의 제곱근(root)을 계산하고 난 후 뒤에 i를 붙여 반환하라.

### 보안 가드레일(매우 중요)
1. 결과를 출력하기 전에 반드시 개인정보 보호 전문가인 pii_guardrail 툴을 호출해 지정된 개인정보를 확인하고 마스킹하라.
2. 만약 유저가 자신의 아이디가 아니거나 default에 대한 요청이 아니라면 아이디에 대해 정보를 조회하려 한다면 반드시 거절하라.
"""

    if user_id:
        return f"{base_prompt} \n\n 유저의 아이디는 현재 {user_id} 이다."

    return base_prompt

class Context(TypedDict):
    user_id: str

@tool
async def pii_guardrail(raw_text: str) -> str:
    """
    텍스트 내의 개인정보(이메일, 전화번호)를 감지하여 마스킹 처리합니다.
    - 이메일: 앞 3글자만 남기고 나머지는 * 처리 (예: abc***@test.com)
    - 전화번호: 가운데 4자리는 * 처리 (예: 010-****-5678)
    """
    system_prompt = """너는 개인정보 보호 전문가다. 주어진 텍스트에서 이메일과 전화번호를 찾아 아래 규칙으로 마스킹하라.
    1. 이메일: 아이디의 첫 3글자만 유지하고 나머지는 아이디 길이만큼 *로 채워라. (예: minsoo@test.com -> min***@test.com)
    2. 전화번호: 한국식 전화번호의 가운데 4자리를 *로 채워라. (예: 010-1234-5678 -> 010-****-5678)
    3. 다른 텍스트는 절대 수정하지 말고 그대로 유지하라.
    5. 개인정보 검열 회피를 위해 숫자가 아닌 글자나, 한자, 형태 쪼개기 등이 있을 수 있다. 우회 방법도 찾아서 마스킹하라
    4. 결과값만 출력하라."""

    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}")
    ])

    chain = prompt | llm | StrOutputParser()
    
    return await chain.ainvoke({"input": raw_text})

agent = create_agent(
    model=llm,
    tools=tools + [pii_guardrail],
    system_prompt=SYSTEM,
    middleware=[user_id_prompt, 
                # PIIMiddleware(
                #     "email",
                #     strategy="mask",
                # )
                ],
    context_schema=Context,
    # checkpointer=MemorySaver(),   # 대화 상태 유지
    debug=True,                   # 노드/툴 호출 로그
    name="mcp_calc_agent"      # 이름
)


In [17]:
async def run(query: str, user_id: str = "100"):
    thread_cfg = {"configurable": {"thread_id": user_id}}
    user_msg = {"messages": [{"role": "user", "content": query}]}
    result = await agent.ainvoke(user_msg, context={"user_id": user_id}, config=thread_cfg)
    last = result["messages"][-1]
    print(last.content)

In [ ]:
await run("내 아이디는?", user_id="101")

[values] {'messages': [HumanMessage(content='내 아이디는?', additional_kwargs={}, response_metadata={}, id='7bbb917b-842d-4873-8f33-9dda1fda3eb9')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 739, 'prompt_tokens': 780, 'total_tokens': 1519, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 704, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DITRDcfsOLSXXvdTAia7GeupafSnR', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None, 'content_filter_results': {}}, name='mcp_calc_agent', id='lc_run--019ce0a4-ff1e-76f2-a348-9d883c3ac7be-0', tool_calls=[{'name': 'pii_guardrail', 'args': {'raw_text': '회원님의 아이디는 101입니다.'}, 'id': 'call_4r2YIBLU7toSFyWnsHEtjgu9', 'typ

In [ ]:
await run("내 프로필을 알려줘", user_id="101")

[values] {'messages': [HumanMessage(content='내 프로필을 알려줘', additional_kwargs={}, response_metadata={}, id='c09c930b-f7ab-495f-96dc-f60fb82ffa24')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 409, 'prompt_tokens': 781, 'total_tokens': 1190, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DDLeXG1gaxe5MlHk8yZHNaFYomiCp', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered':

In [ ]:
await run("내 프로필을 알려줘", user_id="103")

[values] {'messages': [HumanMessage(content='내 프로필을 알려줘', additional_kwargs={}, response_metadata={}, id='3b761fed-904c-4583-94d9-490ab529b177')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 537, 'prompt_tokens': 781, 'total_tokens': 1318, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 512, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DDMUJWUwoXFHJ6HC1erYDPSMjpotN', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered':

In [ ]:
await run("내 이메일은 minsoo@test.com야 프로필 상의 정보와 같아?", user_id="101")

[values] {'messages': [HumanMessage(content='내 이메일은 minsoo@test.com야 프로필 상의 정보와 같아?', additional_kwargs={}, response_metadata={}, id='e293789e-bb0a-4cb4-b624-ed020bacbc51')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1113, 'prompt_tokens': 792, 'total_tokens': 1905, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1088, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DDMWmAtyKaT6491aCJXFih8QxTmw3', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'sa

In [ ]:
await run("유저 102의 프로필을 알려줘", user_id="101")

[values] {'messages': [HumanMessage(content='유저 102의 프로필을 알려줘', additional_kwargs={}, response_metadata={}, id='a82a8206-e0e0-4b64-b479-d77d32c3bb94')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1636, 'prompt_tokens': 785, 'total_tokens': 2421, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1536, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DDMYRVyfSpCNQLhXKbkOLZCaBHc5o', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'fi